In [1]:
import kagglehub
import pandas as pd

/Users/cmkl/spring-2/AIC-601/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download the latest version of the dataset
path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")

print("Path to dataset files:", path)

# Assuming the dataset is in CSV format and the main file is named 'anime.csv'
dataset_path = f"{path}/anime.csv"

# Load the dataset into a pandas DataFrame
df = pd.read_csv(dataset_path)

# Display the first few rows of the dataset to understand its structure
print(df.head())

# Drop the 'episodes' and 'members' columns
df = df.drop(columns=['episodes', 'members'])
df["genre"] = df["genre"].fillna("")  # Replace NaN with an empty string
df["genre"] = df["genre"].astype(str)  # Ensure all values are strings

# Display the first few rows of the preprocessed dataset
print(df.head())

# Optionally, save the preprocessed dataset to a new CSV file
preprocessed_path = f"anime_preprocessed.csv"
df.to_csv(preprocessed_path, index=False)

print(f"Preprocessed dataset saved to {preprocessed_path}")

Path to dataset files: /Users/cmkl/.cache/kagglehub/datasets/CooperUnion/anime-recommendations-database/versions/1
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
   anime_id                    

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
df = pd.read_csv("anime_preprocessed.csv")

# Preprocessing: Handle missing genres properly
df["genre"] = df["genre"].fillna("")  # Replace NaN with an empty string
df["genre"] = df["genre"].astype(str)  # Ensure all values are strings

# TF-IDF Vectorization on genres
vectorizer = TfidfVectorizer(stop_words="english")
genre_matrix = vectorizer.fit_transform(df["genre"])

# Compute Cosine Similarity Matrix for genres
cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

# Function to recommend anime
def recommend_anime(title, top_n=5):
    # Check if anime exists
    if title not in df["name"].values:
        return f"Anime '{title}' not found in the dataset."

    # Get index of the given anime
    idx = df[df["name"] == title].index[0]

    # Get the type of the anime (TV, Movie, OVA, etc.)
    anime_type = df.loc[idx, "type"]

    # Get similarity scores for all animes
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Filter only animes with the same type
    filtered_animes = [
        (i, score) for i, score in sim_scores if df.loc[i, "type"] == anime_type
    ]

    # Sort by similarity score (descending)
    filtered_animes = sorted(filtered_animes, key=lambda x: x[1], reverse=True)

    # Get top similar animes (excluding the input anime itself)
    top_animes = filtered_animes[1:top_n + 1]  # Skip first item (itself)

    # Retrieve anime indices
    anime_indices = [i[0] for i in top_animes]

    # Get recommended animes and sort by rating (descending)
    recommended_animes = df.iloc[anime_indices].sort_values(by="rating", ascending=False)

    return recommended_animes[["name", "genre", "type", "rating"]]

In [7]:
# Change the anime title to get recommendations for different animes
recommended = recommend_anime("Hibike! Euphonium", top_n=5)
print(recommended)

                     name                 genre type  rating
492     Hibike! Euphonium  Drama, Music, School   TV    8.03
3232      Tribe Cool Crew         Music, School   TV    7.05
3281      Wake Up, Girls!          Drama, Music   TV    7.04
4123  Lemon Angel Project          Drama, Music   TV    6.80
5565    Sasurai no Taiyou          Drama, Music   TV    6.42
